# 📒 QuickNotes — PDF/TXT → summary, key points, concepts, definitions, formulas, related topics, mind map

**How this notebook works**
1. Run the cells top to bottom (setup → core → app).
2. Use the interactive panel below the *app* cell. **Nothing is pre-filled: every result is generated from what you upload or type.** Each result shows a *Provenance* box (file name, size, SHA-256 fingerprint) so you can prove it came from your input.
3. Failures never crash the notebook: a bad file or API hiccup gives a readable message and you can simply correct the input and press the button again.
4. No widgets (rare)? Run `run_cli()` in the fallback cell for an `input()`-based version.

## 1. Setup

In [ ]:
# One-time setup (about 1-3 minutes in Colab). Safe to re-run.
import subprocess, sys, shutil
def pip(*pkgs):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], capture_output=True, text=True)
    if r.returncode: print("pip warning for", pkgs, "->", r.stderr[-300:])
pip("anthropic", "pypdf", "ipywidgets", "numpy", "scikit-learn", "sentence-transformers", "pytesseract", "pdf2image", "graphviz")
if shutil.which("tesseract") is None or shutil.which("pdftoppm") is None:      # OCR for scanned PDFs
    subprocess.run("apt-get -qq update && apt-get -qq install -y tesseract-ocr poppler-utils graphviz", shell=True, capture_output=True)
print("OCR available:", shutil.which("tesseract") is not None, "| setup done")

## 2. Core utilities (validation, AI calls with retry/back-off, document reading with OCR fallback)

In [ ]:
import os, sys, io, re, json, time, math, hashlib, getpass, subprocess, tempfile, html, traceback, difflib, warnings
from pathlib import Path
from dataclasses import dataclass, field
warnings.filterwarnings("ignore")

# ------------------------------------------------------------------ config
MAX_FILE_MB = 25                                   # reject bigger uploads
MODEL = os.environ.get("CLAUDE_MODEL", "claude-sonnet-5-5")
STORE_DIR = os.environ.get("STUDYAI_STORE", "studyai_store")   # point this at Google Drive to persist across sessions

class InputError(Exception):
    """A problem with the user's input that the user can fix (bad file, empty, unsupported...)."""

class AIError(Exception):
    """A problem talking to the AI backend (bad key, rate limit, network...)."""

def sha256(b: bytes) -> str:
    return hashlib.sha256(b).hexdigest()

def esc(s) -> str:
    return html.escape(str(s))

# ------------------------------------------------------------------ AI backend (swap this one function to change provider)
_client = None

def _get_api_key():
    key = os.environ.get("ANTHROPIC_API_KEY")
    if not key:
        try:
            from google.colab import userdata          # Colab "Secrets" panel
            key = userdata.get("ANTHROPIC_API_KEY")
        except Exception:
            key = None
    if not key:
        key = getpass.getpass("Paste your Anthropic API key (input hidden): ").strip()
    if key:
        os.environ["ANTHROPIC_API_KEY"] = key
    return key

def reset_api_key():
    """Call this if you pasted a wrong key."""
    global _client
    _client = None
    os.environ.pop("ANTHROPIC_API_KEY", None)

def llm(prompt: str, system: str = "", max_tokens: int = 3000, retries: int = 4) -> str:
    """One text-in/text-out call with retry + back-off for rate limits and transient failures."""
    global _client
    import anthropic
    if _client is None:
        key = _get_api_key()
        if not key:
            raise AIError("No API key provided. Run the API-key cell again.")
        _client = anthropic.Anthropic(api_key=key, max_retries=0)
    delay = 3
    last = None
    for attempt in range(retries + 1):
        try:
            r = _client.messages.create(
                model=MODEL, max_tokens=max_tokens,
                system=system or "You are a careful, accurate study assistant.",
                messages=[{"role": "user", "content": prompt}])
            return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")
        except anthropic.AuthenticationError:
            reset_api_key()
            raise AIError("The API key was rejected. Run reset_api_key() and re-enter a valid key.")
        except anthropic.RateLimitError as e:
            last = "rate limit reached"
        except (anthropic.APIConnectionError, anthropic.APITimeoutError) as e:
            last = "network problem"
        except anthropic.APIStatusError as e:
            if e.status_code in (429, 500, 502, 503, 529):
                last = f"temporary server error ({e.status_code})"
            else:
                raise AIError(f"API error {e.status_code}: {getattr(e, 'message', e)}")
        if attempt < retries:
            print(f"   ... {last}; retrying in {delay}s ({attempt + 1}/{retries})")
            time.sleep(delay)
            delay *= 2
    raise AIError(f"AI backend unavailable after {retries + 1} attempts ({last}). Wait a minute and try again.")

def parse_json_loose(text: str):
    t = text.strip()
    t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.S).strip()
    try:
        return json.loads(t)
    except Exception:
        pass
    a, b = t.find("{"), t.rfind("}")
    if a != -1 and b > a:
        return json.loads(t[a:b + 1])
    raise ValueError("no JSON object found")

def llm_json(prompt: str, system: str = "", max_tokens: int = 3500) -> dict:
    raw = llm(prompt, system, max_tokens)
    try:
        return parse_json_loose(raw)
    except Exception:
        fixed = llm("The following was supposed to be a single valid JSON object but is malformed. "
                    "Return ONLY the corrected JSON object, nothing else.\n\n" + raw, "You repair JSON.", max_tokens)
        try:
            return parse_json_loose(fixed)
        except Exception as e:
            raise AIError(f"The AI returned a response that could not be parsed ({e}). Try again.")

# ------------------------------------------------------------------ validation
DOC_EXT = {".pdf", ".txt"}
AUDIO_EXT = {".mp3", ".wav", ".m4a"}

def validate_upload(name: str, data: bytes, allowed: set, max_mb: float = None) -> str:
    """Return the lower-case extension, or raise InputError with a fixable message."""
    max_mb = MAX_FILE_MB if max_mb is None else max_mb
    ext = Path(name).suffix.lower()
    if ext not in allowed:
        raise InputError(f"'{name}': unsupported format '{ext or 'none'}'. Allowed: {', '.join(sorted(allowed))}")
    if data is None or len(data) == 0:
        raise InputError(f"'{name}' is empty (0 bytes). Choose a different file.")
    if len(data) > max_mb * 1024 * 1024:
        raise InputError(f"'{name}' is {len(data) / 1048576:.1f} MB; the limit is {max_mb} MB. Split or compress it.")
    if ext == ".pdf" and b"%PDF" not in data[:1024]:
        raise InputError(f"'{name}' does not look like a real PDF (wrong header). It may be a renamed file.")
    if ext == ".wav" and not (data[:4] == b"RIFF" and data[8:12] == b"WAVE"):
        raise InputError(f"'{name}' does not look like a real WAV file (wrong header).")
    return ext

# ------------------------------------------------------------------ small text helpers
def norm(s: str) -> str:
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s]", " ", str(s).lower())).strip()

def words(s: str) -> list:
    return re.findall(r"\w+", str(s).lower())

def jaccard(a: str, b: str) -> float:
    A, B = set(words(a)), set(words(b))
    return len(A & B) / max(1, len(A | B))

def split_text(t: str, size: int = 900, overlap: int = 150) -> list:
    t = t.strip()
    out, i = [], 0
    while i < len(t):
        j = min(len(t), i + size)
        if j < len(t):
            k = max(t.rfind(". ", i + size // 2, j), t.rfind("\n", i + size // 2, j))
            if k != -1:
                j = k + 1
        out.append(t[i:j].strip())
        if j >= len(t):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if len(c) >= 30 or len(out) == 1]

# ------------------------------------------------------------------ notebook UI helpers
try:
    import ipywidgets as W
    from IPython.display import display, clear_output, HTML, Markdown
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    try:
        from IPython.display import display, clear_output, HTML, Markdown
    except Exception:
        display = print
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()     # lets ipywidgets (incl. FileUpload) work in Colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

CSS = """<style>
.sn-card{border:1px solid #8884;border-radius:10px;padding:12px 16px;margin:8px 0}
.sn-ok{background:#2e7d3218;border-left:4px solid #2e7d32}.sn-warn{background:#ef6c0018;border-left:4px solid #ef6c00}
.sn-bad{background:#c6282818;border-left:4px solid #c62828}.sn-info{background:#1565c018;border-left:4px solid #1565c0}
.sn-mark{background:#ffeb3b88;padding:0 2px}.sn-unc{background:#ff980055;border-bottom:2px dotted #e65100;padding:0 2px}
.sn-small{font-size:.85em;opacity:.8}.sn-table td,.sn-table th{padding:4px 10px;border-bottom:1px solid #8883;text-align:left;vertical-align:top}
</style>"""

def show_html(s: str):
    display(HTML(CSS + s))

def callout(kind: str, msg: str) -> str:
    icon = {"ok": "✅", "warn": "⚠️", "bad": "⛔", "info": "ℹ️"}[kind]
    return f"<div class='sn-card sn-{kind}'>{icon} {msg}</div>"

def uploaded_items(up) -> list:
    """Normalise ipywidgets FileUpload .value across v7 (dict) and v8 (tuple) -> [(name, bytes)]."""
    v = up.value
    items = []
    if isinstance(v, dict):
        for name, meta in v.items():
            items.append((name, bytes(meta["content"])))
    else:
        for f in v:
            items.append((f["name"], bytes(f["content"])))
    return items

def reset_upload(up):
    for fn in (lambda: up.value.clear(), lambda: setattr(up, "value", ()), lambda: setattr(up, "_counter", 0)):
        try:
            fn()
        except Exception:
            pass

def cli_upload(allowed: set) -> list:
    """Fallback when ipywidgets is unavailable: Colab file picker, else typed paths."""
    if IN_COLAB:
        from google.colab import files
        up = files.upload()
        return [(n, d) for n, d in up.items()]
    raw = input(f"Path(s) to file(s) ({', '.join(sorted(allowed))}), separated by commas: ")
    out = []
    for p in [x.strip() for x in raw.split(",") if x.strip()]:
        try:
            out.append((Path(p).name, Path(p).read_bytes()))
        except Exception as e:
            print(f"⚠️ Cannot read '{p}': {e}")
    return out

def friendly(fn, *a, **k):
    """Run fn; print a readable message instead of crashing the notebook."""
    try:
        return fn(*a, **k)
    except InputError as e:
        show_html(callout("warn", f"<b>Input problem:</b> {esc(e)}<br><span class='sn-small'>Fix the input and try again — nothing else was affected.</span>"))
    except AIError as e:
        show_html(callout("bad", f"<b>AI backend:</b> {esc(e)}<br><span class='sn-small'>Your uploaded data is still loaded; just press the button again.</span>"))
    except Exception as e:
        show_html(callout("bad", f"<b>Unexpected error:</b> {esc(type(e).__name__)}: {esc(e)}<br><span class='sn-small'>The notebook is still usable. Details below.</span>"))
        traceback.print_exc(limit=3)
    return None

In [ ]:
import numpy as np

@dataclass
class LoadedDoc:
    doc_id: str
    name: str
    sha: str
    size: int
    kind: str                      # "pdf" or "txt"
    pages: list                    # [(page_or_section_no, text)]
    student: str = ""
    notes: list = field(default_factory=list)
    ocr_pages: list = field(default_factory=list)

    @property
    def text(self) -> str:
        return "\n\n".join(t for _, t in self.pages)

    @property
    def text_with_pages(self) -> str:
        lab = "Page" if self.kind == "pdf" else "Section"
        return "\n\n".join(f"[{lab} {n}]\n{t}" for n, t in self.pages)

    @property
    def label(self) -> str:
        return "p." if self.kind == "pdf" else "§"

def clean_text(t: str) -> str:
    t = (t or "").replace("\x00", "")
    t = re.sub(r"-\n(\w)", r"\1", t)
    t = re.sub(r"[ \t\f\v]+", " ", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()

def _ocr_pages(data: bytes, page_numbers: list):
    try:
        import pytesseract
        from pdf2image import convert_from_bytes
        pytesseract.get_tesseract_version()
    except Exception:
        return {}, "OCR is not installed (run the install cell, which tries to apt-get install tesseract-ocr + poppler-utils)."
    out, err = {}, None
    for n in page_numbers[:80]:
        try:
            img = convert_from_bytes(data, dpi=200, first_page=n, last_page=n)[0]
            out[n] = pytesseract.image_to_string(img)
        except Exception as e:
            err = f"OCR failed on page {n}: {e}"
    return out, err

def extract_pdf_pages(data: bytes, name: str, use_ocr: bool = True):
    try:
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(data))
        if reader.is_encrypted:
            try:
                ok = reader.decrypt("")
            except Exception:
                ok = 0
            if not ok:
                raise InputError(f"'{name}' is password-protected. Remove the password and upload again.")
        raw = []
        for i, p in enumerate(reader.pages, 1):
            try:
                raw.append((i, clean_text(p.extract_text() or "")))
            except Exception:
                raw.append((i, ""))
    except InputError:
        raise
    except Exception as e:
        raise InputError(f"'{name}' could not be read as a PDF (corrupt or damaged): {e}")
    if not raw:
        raise InputError(f"'{name}' has no pages.")
    notes, ocr_done = [], []
    blank = [n for n, t in raw if len(t) < 25]
    if blank and use_ocr:
        res, err = _ocr_pages(data, blank)
        d = dict(raw)
        for n, t in res.items():
            t = clean_text(t)
            if len(t) >= 25:
                d[n] = t
                ocr_done.append(n)
        raw = sorted(d.items())
        if ocr_done:
            notes.append(f"OCR was used on {len(ocr_done)} page(s): {ocr_done[:15]}{'...' if len(ocr_done) > 15 else ''} — OCR text may contain errors.")
        if err and len(res) < len(blank):
            notes.append(err)
    still_blank = [n for n, t in raw if len(t) < 25]
    if len(still_blank) == len(raw):
        raise InputError(f"'{name}' looks like a scanned/image-only PDF and no text could be extracted"
                         f"{' (OCR unavailable or found nothing)' if use_ocr else ' (OCR is switched off)'}. "
                         "Enable OCR / install tesseract, or upload a text-based PDF.")
    if still_blank:
        notes.append(f"{len(still_blank)} page(s) had no readable text and were skipped: {still_blank[:15]}")
    pages = [(n, t) for n, t in raw if len(t) >= 25]
    return pages, ocr_done, notes

def decode_text(data: bytes, name: str) -> str:
    for enc in ("utf-8-sig", "utf-16", "cp1252", "latin-1"):
        try:
            t = data.decode(enc)
            if t.count("\x00") > len(t) * 0.01:
                continue
            return t
        except Exception:
            continue
    raise InputError(f"'{name}' could not be decoded as text.")

def load_document(name: str, data: bytes, student: str = "", use_ocr: bool = True) -> LoadedDoc:
    ext = validate_upload(name, data, DOC_EXT)
    sha = sha256(data)
    if ext == ".pdf":
        pages, ocr_done, notes = extract_pdf_pages(data, name, use_ocr)
        kind = "pdf"
    else:
        t = clean_text(decode_text(data, name))
        if len(t) < 25:
            raise InputError(f"'{name}' contains almost no text ({len(t)} characters).")
        parts = split_text(t, 3000, 0) or [t]
        pages, ocr_done, notes, kind = list(enumerate(parts, 1)), [], [], "txt"
    total = sum(len(t) for _, t in pages)
    if total < 100:
        raise InputError(f"'{name}' has too little text ({total} characters) to work with.")
    return LoadedDoc(doc_id=sha[:10], name=name, sha=sha, size=len(data), kind=kind, pages=pages,
                     student=(student or "").strip(), notes=notes, ocr_pages=ocr_done)

# ------------------------------------------------------------------ persistence ("previously uploaded documents")
def save_doc(doc: LoadedDoc):
    try:
        Path(STORE_DIR).mkdir(parents=True, exist_ok=True)
        Path(STORE_DIR, f"{doc.doc_id}.json").write_text(json.dumps(doc.__dict__, ensure_ascii=False))
    except Exception as e:
        print(f"(could not save '{doc.name}' to {STORE_DIR}: {e})")

def load_saved_docs() -> list:
    out = []
    for p in sorted(Path(STORE_DIR).glob("*.json")) if Path(STORE_DIR).exists() else []:
        try:
            d = json.loads(p.read_text())
            d["pages"] = [tuple(x) for x in d["pages"]]
            out.append(LoadedDoc(**d))
        except Exception:
            pass
    return out

# ------------------------------------------------------------------ chunking + retrieval
def chunk_doc(doc: LoadedDoc, size: int = 900, overlap: int = 150) -> list:
    chunks = []
    for page, text in doc.pages:
        for j, c in enumerate(split_text(text, size, overlap)):
            chunks.append(dict(doc_id=doc.doc_id, doc_name=doc.name, student=doc.student,
                               page=page, label=doc.label, text=c))
    return chunks

class Index:
    """Dense retrieval (sentence-transformers) with an automatic TF-IDF fallback."""
    def __init__(self):
        self.chunks, self._E, self._tf, self._M = [], None, None, None
        try:
            from sentence_transformers import SentenceTransformer
            self._st = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
            self.backend = "sentence-transformers (dense)"
        except Exception as e:
            self._st, self.backend = None, "TF-IDF (fallback)"
        self.min_relevance = 0.30 if self._st is not None else 0.10     # below this, passages are "not relevant"
        self.support_score = 0.45 if self._st is not None else 0.20

    def has_doc(self, doc_id): return any(c["doc_id"] == doc_id for c in self.chunks)

    def add(self, doc: LoadedDoc):
        if self.has_doc(doc.doc_id):
            return
        new = chunk_doc(doc)
        if self._st is not None and new:
            E = self._st.encode([c["text"] for c in new], normalize_embeddings=True, show_progress_bar=False)
            for c, e in zip(new, E):
                c["emb"] = e
        self.chunks += new
        self._refit()

    def remove(self, doc_id):
        self.chunks = [c for c in self.chunks if c["doc_id"] != doc_id]
        self._refit()

    def _refit(self):
        if not self.chunks:
            self._E = self._M = self._tf = None
            return
        if self._st is not None:
            self._E = np.vstack([c["emb"] for c in self.chunks])
        else:
            from sklearn.feature_extraction.text import TfidfVectorizer
            self._tf = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, stop_words="english")
            self._M = self._tf.fit_transform([c["text"] for c in self.chunks])

    def _scores(self, q):
        if self._st is not None:
            return self._E @ self._st.encode([q], normalize_embeddings=True, show_progress_bar=False)[0]
        return (self._M @ self._tf.transform([q]).T).toarray().ravel()

    def search(self, query, k=5, doc_ids=None, student=None):
        if not self.chunks or not str(query).strip():
            return []
        s = self._scores(query)
        order = np.argsort(-s)
        out = []
        for i in order:
            c = self.chunks[i]
            if doc_ids and c["doc_id"] not in doc_ids:
                continue
            if student and student.strip().lower() not in ("", "all students") and c["student"].lower() != student.strip().lower():
                continue
            r = {k_: v for k_, v in c.items() if k_ != "emb"}
            r["score"] = float(s[i])
            out.append(r)
            if len(out) >= k:
                break
        return out

In [ ]:
# Enter your Anthropic API key once. Order tried: environment variable -> Colab Secrets (ANTHROPIC_API_KEY) -> hidden prompt.
# The AI backend is the single function llm() in the next cell — swap it to use another provider.
_get_api_key()
print("Key loaded. Model:", MODEL)

## 3. QuickNotes app — upload your own file
Upload one or more PDF/TXT files (or paste text), choose **Length** and **Detail**, press **Process**. Change the settings and press **Regenerate** to see the same document summarised differently. Upload another file at any time — no restart needed. Use the *Compare* section to inspect any generated claim next to the exact passage of your original it came from.

In [ ]:
LENGTHS = {"Short (~100 words)": (80, 130), "Medium (~250 words)": (200, 300), "Long (~500 words)": (420, 600)}
DETAIL = {"Brief": dict(kp=5, concepts=5, defs=4, formulas=4),
          "Standard": dict(kp=9, concepts=8, defs=8, formulas=8),
          "Comprehensive": dict(kp=15, concepts=14, defs=14, formulas=14)}

DOCS = {}        # sha -> LoadedDoc  (everything uploaded this session)
RESULTS = {}     # (sha, length, detail) -> generated notes
INDEX = Index()
print(f"Retrieval backend for grounding checks: {INDEX.backend}")

SYSTEM = ("You turn study documents into study material. Use ONLY information in the document. Never invent facts, "
          "definitions or formulas. If the document has no formulas/definitions, return empty lists for them. "
          "'evidence' fields must be exact short quotes copied verbatim from the document.")

SCHEMA = """{
 "title": "short title",
 "summary": "the summary",
 "key_points": [{"point": "...", "evidence": "exact quote from the document (max 25 words)"}],
 "concepts": [{"term": "...", "explanation": "..."}],
 "definitions": [{"term": "...", "definition": "...", "evidence": "exact quote"}],
 "formulas": [{"formula": "as written in the document", "meaning": "...", "evidence": "exact quote"}],
 "related_topics": ["topics NOT covered in the document that are worth studying next"],
 "mind_map": {"name": "central topic", "children": [{"name": "branch", "children": [{"name": "leaf", "children": []}]}]}
}"""

def digest_long(text: str) -> str:
    """Map step for very long documents: condense sections but keep definitions/formulas verbatim."""
    parts = split_text(text, 12000, 0)
    out = []
    for i, p in enumerate(parts, 1):
        print(f"   condensing part {i}/{len(parts)} ...")
        out.append(llm("Condense this part of a document into dense study notes (bullets). Keep every definition, formula, "
                       "number and named concept; copy definitions and formulas VERBATIM. Do not add outside information.\n\n" + p,
                       SYSTEM, 2500))
    return "\n\n".join(out)

def generate_notes(doc: LoadedDoc, length: str, detail: str, force=False) -> dict:
    key = (doc.sha, length, detail)
    if key in RESULTS and not force:
        return RESULTS[key]
    lo, hi = LENGTHS[length]
    d = DETAIL[detail]
    text = doc.text_with_pages
    condensed = False
    if len(text) > 40000:
        text, condensed = digest_long(doc.text), True
    prompt = (f"DOCUMENT{' (condensed notes of a long document)' if condensed else ''}:\n<<<\n{text}\n>>>\n\n"
              f"Produce study material as ONE JSON object with exactly this shape:\n{SCHEMA}\n\n"
              f"Requirements:\n- summary: {lo}-{hi} words, based only on the document.\n"
              f"- key_points: about {d['kp']}; concepts: up to {d['concepts']}; definitions: up to {d['defs']}; "
              f"formulas: up to {d['formulas']} (only formulas actually present).\n"
              f"- mind_map: 3-6 branches, depth 2-3, short labels.\n- Output JSON only.")
    res = llm_json(prompt, SYSTEM, 4500)
    for k_ in ("key_points", "concepts", "definitions", "formulas", "related_topics"):
        res.setdefault(k_, [])
    res.setdefault("summary", "")
    res.setdefault("mind_map", {"name": doc.name, "children": []})
    res["_verify"] = verify_result(doc, res)
    res["_condensed"] = condensed
    RESULTS[key] = res
    return res

def page_of(doc, nq):
    for n, t in doc.pages:
        if nq in norm(t):
            return n
    return None

def verify_item(doc, quote, claim):
    nq, nt = norm(quote), norm(doc.text)
    if nq and len(nq) > 8 and nq in nt:
        return dict(status="verbatim", score=1.0, page=page_of(doc, nq), passage=quote)
    hits = INDEX.search((claim or "") + " " + (quote or ""), k=1, doc_ids={doc.doc_id})
    if not hits:
        return dict(status="unsupported", score=0.0, page=None, passage="")
    h = hits[0]
    st = "supported" if h["score"] >= INDEX.support_score else "weak"
    return dict(status=st, score=h["score"], page=h["page"], passage=h["text"])

def verify_result(doc, res) -> list:
    items = []
    for i, x in enumerate(res["key_points"], 1):
        items.append(dict(kind="Key point", label=f"Key point {i}", claim=x.get("point", ""), quote=x.get("evidence", "")))
    for x in res["definitions"]:
        items.append(dict(kind="Definition", label=f"Definition: {x.get('term', '')}", claim=f"{x.get('term', '')}: {x.get('definition', '')}", quote=x.get("evidence", "")))
    for x in res["formulas"]:
        f = x.get("formula", "")
        v = verify_item(doc, x.get("evidence", "") or f, x.get("meaning", ""))
        compact = re.sub(r"\s+", "", f)
        if compact and compact in re.sub(r"\s+", "", doc.text):
            v.update(status="verbatim", score=1.0)
        items.append(dict(kind="Formula", label=f"Formula: {f[:40]}", claim=f"{f} — {x.get('meaning', '')}", quote=x.get("evidence", "") or f, **{"_pre": v}))
    for it in items:
        it["check"] = it.pop("_pre", None) or verify_item(doc, it["quote"], it["claim"])
    return items

# ------------------------------------------------------------------ mind map
def mindmap_text(node, prefix="", last=True, root=True) -> str:
    line = (node.get("name", "?") + "\n") if root else (prefix + ("└── " if last else "├── ") + node.get("name", "?") + "\n")
    kids = node.get("children", []) or []
    for i, c in enumerate(kids):
        line += mindmap_text(c, prefix + ("" if root else ("    " if last else "│   ")), i == len(kids) - 1, False)
    return line

def mindmap_graph(node):
    try:
        import graphviz
        g = graphviz.Digraph(graph_attr=dict(rankdir="LR", bgcolor="transparent"),
                             node_attr=dict(shape="box", style="rounded,filled", fillcolor="#e8f0fe", fontname="Helvetica"))
        cnt = [0]
        def walk(n, parent=None):
            nid = f"n{cnt[0]}"; cnt[0] += 1
            g.node(nid, str(n.get("name", "?"))[:40], fillcolor="#c5d9fb" if parent is None else "#e8f0fe")
            if parent:
                g.edge(parent, nid)
            for c in n.get("children", []) or []:
                walk(c, nid)
        walk(node)
        return g
    except Exception:
        return None

# ------------------------------------------------------------------ rendering
BADGE = {"verbatim": ("✅ verbatim in source", "ok"), "supported": ("🟢 supported by source passage", "ok"),
         "weak": ("🟠 weakly supported — check", "warn"), "unsupported": ("🔴 not found in source", "bad")}

def render(doc: LoadedDoc, res: dict, length: str, detail: str):
    v = res["_verify"]
    n_ok = sum(1 for x in v if x["check"]["status"] in ("verbatim", "supported"))
    words_out = len(words(res["summary"]))
    h = [f"<h2>{esc(res.get('title') or doc.name)}</h2>"]
    h.append("<div class='sn-card sn-info'><b>Provenance</b> — generated from your uploaded file: "
             f"<code>{esc(doc.name)}</code> · {doc.size:,} bytes · SHA-256 <code>{doc.sha[:16]}…</code> · "
             f"{len(doc.pages)} {'pages' if doc.kind == 'pdf' else 'sections'} · {len(doc.text):,} characters extracted"
             f"{' · long document processed in parts' if res['_condensed'] else ''}<br>"
             f"Settings: <b>{esc(length)}</b>, <b>{esc(detail)}</b> · summary = {words_out} words "
             f"({words_out / max(1, len(words(doc.text))) * 100:.1f}% of the original) · "
             f"grounding check: <b>{n_ok}/{len(v)}</b> claims traced to the source</div>")
    for n in doc.notes:
        h.append(callout("warn", esc(n)))
    h.append(f"<h3>Summary</h3><div class='sn-card'>{esc(res['summary'])}</div>")
    def badge(item):
        t, k = BADGE[item["check"]["status"]]
        pg = f" · {doc.label}{item['check']['page']}" if item["check"]["page"] else ""
        return f"<span class='sn-small'>{t}{pg}</span>"
    vi = {(x["kind"], x["claim"]): x for x in v}
    h.append("<h3>Key points</h3><ol>")
    for x in res["key_points"]:
        it = vi.get(("Key point", x.get("point", "")))
        h.append(f"<li>{esc(x.get('point'))} {badge(it) if it else ''}</li>")
    h.append("</ol><h3>Concepts</h3><ul>")
    for x in res["concepts"]:
        h.append(f"<li><b>{esc(x.get('term'))}</b> — {esc(x.get('explanation'))}</li>")
    h.append("</ul><h3>Definitions</h3>")
    h.append("<ul>" + "".join(
        f"<li><b>{esc(x.get('term'))}</b>: {esc(x.get('definition'))} "
        f"{badge(vi[('Definition', x.get('term', '') + ': ' + x.get('definition', ''))]) if ('Definition', x.get('term', '') + ': ' + x.get('definition', '')) in vi else ''}</li>"
        for x in res["definitions"]) + "</ul>" if res["definitions"] else "<p class='sn-small'>No explicit definitions found in this document.</p>")
    h.append("<h3>Formulas</h3>")
    h.append("<ul>" + "".join(
        f"<li><code>{esc(x.get('formula'))}</code> — {esc(x.get('meaning'))}</li>" for x in res["formulas"]) + "</ul>"
        if res["formulas"] else "<p class='sn-small'>No formulas found in this document.</p>")
    h.append("<h3>Related topics <span class='sn-small'>(not in the document — suggestions for further study)</span></h3><ul>"
             + "".join(f"<li>{esc(t)}</li>" for t in res["related_topics"]) + "</ul>")
    show_html("".join(h))
    print("Mind map (text):\n" + mindmap_text(res["mind_map"]))
    g = mindmap_graph(res["mind_map"])
    if g is not None:
        try:
            display(g)
        except Exception:
            pass

def source_context(doc, passage, quote, page):
    txt = dict(doc.pages).get(page, "") if page else ""
    if not txt:
        txt = passage
    t = esc(txt)
    q = esc(quote.strip()) if quote else ""
    if q and q in t:
        t = t.replace(q, f"<span class='sn-mark'>{q}</span>", 1)
    return t

# ------------------------------------------------------------------ pipeline
STATE = {"current": None}   # sha of doc currently displayed

def process_items(items, use_ocr, length, detail, student=""):
    last = None
    for name, data in items:
        try:
            doc = load_document(name, data, student, use_ocr)
            if doc.sha in DOCS:
                show_html(callout("info", f"<code>{esc(name)}</code> was already processed (same content) — reusing it."))
            DOCS.setdefault(doc.sha, doc)
            INDEX.add(DOCS[doc.sha]); save_doc(DOCS[doc.sha])
            last = DOCS[doc.sha]
            print(f"✔ extracted '{name}'  ({len(last.text):,} characters)")
        except InputError as e:
            show_html(callout("warn", f"<b>Skipped:</b> {esc(e)}"))
        except Exception as e:
            show_html(callout("bad", f"<b>Skipped '{esc(name)}':</b> {esc(e)}"))
    return last

def show_doc(sha, length, detail, force=False):
    doc = DOCS[sha]
    print(f"Generating notes for '{doc.name}' ...")
    res = generate_notes(doc, length, detail, force)
    STATE["current"] = sha
    render(doc, res, length, detail)
    return doc, res

# ------------------------------------------------------------------ widgets
if HAS_WIDGETS:
    up = W.FileUpload(accept=".pdf,.txt", multiple=True, description="Upload PDF/TXT")
    ocr_cb = W.Checkbox(value=True, description="Use OCR for scanned pages", indent=False)
    paste = W.Textarea(placeholder="…or paste your own text here (optional)", layout=W.Layout(width="95%", height="70px"))
    dd_len = W.Dropdown(options=list(LENGTHS), value="Medium (~250 words)", description="Length:")
    dd_det = W.Dropdown(options=list(DETAIL), value="Standard", description="Detail:")
    dd_doc = W.Dropdown(options=[], description="Document:", layout=W.Layout(width="450px"))
    b_proc = W.Button(description="Process uploaded files", button_style="primary", icon="cogs")
    b_regen = W.Button(description="Regenerate with these settings", icon="refresh")
    out = W.Output()
    cmp_out = W.Output()
    cmp_dd = W.Dropdown(options=[], description="Compare:", layout=W.Layout(width="600px"))
    find_t = W.Text(placeholder="search the ORIGINAL text…", layout=W.Layout(width="300px"))
    b_find = W.Button(description="Find in original", icon="search")

    def refresh_docs(select=None):
        dd_doc.options = [(f"{d.name} ({d.sha[:6]})", s) for s, d in DOCS.items()]
        if select:
            dd_doc.value = select

    def refresh_compare():
        sha = STATE["current"]
        if not sha:
            return
        res = RESULTS.get((sha, dd_len.value, dd_det.value))
        if res:
            cmp_dd.options = [(f"{x['label']}  [{x['check']['status']}]", i) for i, x in enumerate(res["_verify"])]

    def on_process(_):
        items = uploaded_items(up)
        if paste.value.strip():
            items.append(("pasted_text.txt", paste.value.encode("utf-8")))
        with out:
            clear_output()
            if not items:
                show_html(callout("warn", "No file selected. Click <b>Upload PDF/TXT</b>, choose file(s), then press this button."))
                return
            def run():
                doc = process_items(items, ocr_cb.value, dd_len.value, dd_det.value)
                if doc:
                    refresh_docs(doc.sha)
                    show_doc(doc.sha, dd_len.value, dd_det.value)
                    refresh_compare()
            friendly(run)
        reset_upload(up)

    def on_regen(_):
        with out:
            clear_output()
            if not dd_doc.value:
                show_html(callout("warn", "Process a file first."))
                return
            friendly(lambda: (show_doc(dd_doc.value, dd_len.value, dd_det.value, force=True), refresh_compare()))

    def on_pick(change):
        if change["name"] == "value" and change["new"]:
            with out:
                clear_output()
                friendly(lambda: (show_doc(change["new"], dd_len.value, dd_det.value), refresh_compare()))

    def on_compare(change):
        if change["name"] != "value" or change["new"] is None or not STATE["current"]:
            return
        doc = DOCS[STATE["current"]]
        res = RESULTS.get((STATE["current"], dd_len.value, dd_det.value))
        if not res:
            return
        x = res["_verify"][change["new"]]
        c = x["check"]
        t, k = BADGE[c["status"]]
        with cmp_out:
            clear_output()
            show_html(f"<div class='sn-card'><b>Generated ({esc(x['kind'])}):</b> {esc(x['claim'])}</div>"
                      f"<div class='sn-card sn-{k}'><b>{t}</b> (similarity {c['score']:.2f}"
                      f"{', ' + doc.label + str(c['page']) if c['page'] else ''})<br><br>"
                      f"<b>Original text{' (page/section)' if c['page'] else ''}:</b><br>"
                      f"<div style='white-space:pre-wrap'>{source_context(doc, c['passage'], x['quote'], c['page'])}</div></div>")

    def on_find(_):
        with cmp_out:
            clear_output()
            if not STATE["current"] or not find_t.value.strip():
                print("Process a document and type a search term.")
                return
            doc, term = DOCS[STATE["current"]], find_t.value.strip()
            hits = [(m.start()) for m in re.finditer(re.escape(term), doc.text, flags=re.I)][:8]
            if not hits:
                show_html(callout("info", f"'{esc(term)}' does not appear in the original text."))
                return
            body = "".join(
                "<div class='sn-card'>…" + esc(doc.text[max(0, h - 140):h]) + "<span class='sn-mark'>" + esc(doc.text[h:h + len(term)]) + "</span>"
                + esc(doc.text[h + len(term):h + len(term) + 140]) + "…</div>" for h in hits)
            show_html(f"<b>{len(hits)} match(es) shown</b>" + body)

    b_proc.on_click(on_process); b_regen.on_click(on_regen); b_find.on_click(on_find)
    dd_doc.observe(on_pick, names="value"); cmp_dd.observe(on_compare, names="value")
    display(W.VBox([
        W.HTML("<h3>📒 QuickNotes — test with your own file</h3>"),
        W.HBox([up, ocr_cb]), paste,
        W.HBox([dd_len, dd_det]), W.HBox([b_proc, b_regen]), dd_doc, out,
        W.HTML("<h3>🔍 Compare generated output with the original</h3>"),
        cmp_dd, W.HBox([find_t, b_find]), cmp_out]))
else:
    print("ipywidgets not available — run  run_cli()  in the next cell instead.")

def run_cli():
    """Plain input()-based alternative (works in any Jupyter / Colab)."""
    items = cli_upload(DOC_EXT)
    length = list(LENGTHS)[int(input(f"Length {list(enumerate(LENGTHS))} : ") or 1)]
    detail = list(DETAIL)[int(input(f"Detail {list(enumerate(DETAIL))} : ") or 1)]
    doc = process_items(items, True, length, detail)
    if doc:
        friendly(show_doc, doc.sha, length, detail)

### Fallback (only if widgets don't show)
`run_cli()` asks for a file and settings with plain `input()`.

In [ ]:
# run_cli()

## Custom Input Testing and Validation

This section lets you verify, with your own inputs, that the notebook behaves as required. The automated cell tests the *input guards* using files created on the fly; the manual table lists the behaviours to check with your real files.

### Manual acceptance tests (use your own files)
| # | Do this | You should see |
|---|---|---|
| 1 | Upload a PDF → *Process* | Summary/key points/etc. built from **that** PDF; Provenance box shows its name, size, SHA-256, page count |
| 2 | Upload a *different* PDF | All content changes to the new document; old one stays selectable in the **Document** dropdown |
| 3 | Switch **Length** / **Detail** → *Regenerate* | Summary length and number of key points/concepts change; same document |
| 4 | Pick any item in **Compare** | The generated claim beside the original passage (quote highlighted) + a verbatim / supported / weak badge |
| 5 | Type a word in **Find in original** | Matching excerpts from your file (or "does not appear") |
| 6 | Upload an empty file, a `.docx`, a renamed/corrupt PDF, a scanned PDF | A clear warning for each; the notebook keeps working |

In [ ]:
def run_validation_tests():
    """Checks the input validators with files built on the fly (these test the *guards*, not the outputs)."""
    from pypdf import PdfWriter
    results = []
    def expect_error(label, fn):
        try:
            fn()
            results.append((label, "❌ accepted (should have been rejected)"))
        except InputError as e:
            results.append((label, f"✅ rejected: {str(e)[:110]}"))
        except Exception as e:
            results.append((label, f"❌ crashed with {type(e).__name__}: {e}"))
    blank = io.BytesIO(); w = PdfWriter(); w.add_blank_page(300, 300); w.write(blank)
    expect_error("empty .txt file", lambda: load_document("empty.txt", b""))
    expect_error("whitespace-only .txt", lambda: load_document("space.txt", b"   \n  \n"))
    expect_error("empty .pdf file", lambda: load_document("empty.pdf", b""))
    expect_error("unsupported format (.docx)", lambda: load_document("notes.docx", b"PK\x03\x04 something"))
    expect_error("renamed/corrupt PDF", lambda: load_document("fake.pdf", b"this is not a pdf at all" * 20))
    expect_error("truncated PDF", lambda: load_document("cut.pdf", b"%PDF-1.4\n1 0 obj\n<<"))
    expect_error("oversized file (limit forced to 0.001 MB)", lambda: validate_upload("big.txt", b"x" * 5000, DOC_EXT, max_mb=0.001))
    expect_error("image-only / scanned-style PDF (blank page, OCR on)", lambda: load_document("scan.pdf", blank.getvalue(), use_ocr=True))
    try:
        d = load_document("ok.txt", ("Photosynthesis converts light energy into chemical energy. " * 6).encode())
        results.append(("valid .txt", f"✅ accepted ({len(d.text)} chars)"))
    except Exception as e:
        results.append(("valid .txt", f"❌ {e}"))
    rows = "".join(f"<tr><td>{esc(a)}</td><td>{esc(b)}</td></tr>" for a, b in results)
    show_html(f"<table class='sn-table'><tr><th>Test</th><th>Result</th></tr>{rows}</table>")
    return all(r[1].startswith("✅") for r in results)

def uniqueness_report(sha_a, sha_b, length, detail):
    """After processing two different documents: do the outputs actually differ?"""
    a, b = RESULTS.get((sha_a, length, detail)), RESULTS.get((sha_b, length, detail))
    if not a or not b:
        print("Generate notes for both documents with the same Length/Detail settings first.")
        return
    j = jaccard(a["summary"], b["summary"])
    verdict = "✅ outputs differ as expected" if j < 0.5 else "⚠️ summaries are suspiciously similar — are the documents on the same topic?"
    show_html(callout("ok" if j < 0.5 else "warn", f"Summary word-overlap between <b>{esc(DOCS[sha_a].name)}</b> and <b>{esc(DOCS[sha_b].name)}</b>: {j:.2f} — {verdict}"))


run_validation_tests()

**Cross-document check** — after processing two different files with the same Length/Detail, paste their short hashes (shown in the *Document* dropdown) to confirm the outputs differ:

In [ ]:
print([(d.name, s) for s, d in DOCS.items()])
# uniqueness_report(FULL_SHA_A, FULL_SHA_B, 'Medium (~250 words)', 'Standard')   # use full keys from the list above
shas = list(DOCS)
if len(shas) >= 2:
    uniqueness_report(shas[-2], shas[-1], dd_len.value if HAS_WIDGETS else 'Medium (~250 words)', dd_det.value if HAS_WIDGETS else 'Standard')